# Benchmark owner

You hold a private prompt set. This notebook verifies the enclave, uploads the prompts over the attestation-pinned channel, approves the run and reads the results. You never see the model owner's adapter.

Setup once: install [uv](https://docs.astral.sh/uv/getting-started/installation/) and the [`tinfoil` CLI](https://docs.tinfoil.sh/containers/cli), then from the repo root run `uv run --group notebooks jupyter notebook notebooks/`. Create your key with `dbe keygen --party benchmark-owner` and have the operator put the printed public key into `tinfoil-config.yml`. Set `DBE_ENCLAVE` to your enclave before starting Jupyter, or edit the first code cell.

In [ ]:
PARTY = "benchmark-owner"
import os, subprocess, json
from pathlib import Path
if Path.cwd().name == "notebooks":       # run from the repo root so bench/ paths resolve
    os.chdir("..")
ENCLAVE = os.environ.get("DBE_ENCLAVE", "dbe.tinfoil.containers.tinfoil.dev")
REPO = os.environ.get("DBE_REPO", "tinfoilsh/double-blind-eval")
TAG = os.environ.get("DBE_TAG", "v0.1.0")
os.environ.update(DBE_ENCLAVE=ENCLAVE, DBE_REPO=REPO, DBE_PARTY=PARTY)

def dbe(*args):
    """Run a dbe command and print its output."""
    proc = subprocess.run(["dbe", *args], capture_output=True, text=True)
    print(proc.stdout or proc.stderr)
    return proc

## 1. Verify the enclave

In [ ]:
dbe("verify")

## 2. Prepare and upload the prompt set

The sample below is the MLCommons AILuminate demo set, cut to the first prompt per hazard exactly as OpenMined's demo did. Any CSV with `prompt_text` (or JSONL with `prompt` and optional `expected`) works.

In [ ]:
!bash bench/fetch_ailuminate_demo.sh bench/ailuminate_demo_sample.csv
BENCH = os.environ.get("DBE_BENCH", "bench/ailuminate_demo_sample.csv")
dbe("benchmark", "upload", BENCH)

## 3. Review and approve the run manifest

In [ ]:
dbe("manifest")

In [ ]:
dbe("approve")

## 4. Read the results

Per prompt: completion, time to first token, decode tokens per second. If the benchmark had an `expected` column the exact-match score is included. The receipt is signed by the enclave and carries both approvals.

In [ ]:
dbe("run", "--wait")
dbe("results", "--out", "results.json", "--show")
receipt = json.load(open("results.json"))["receipt"]
json.dump(receipt, open("receipt.json", "w"), indent=2)
dbe("receipt", "verify", "receipt.json", "--tag", TAG)